# flip velocity fits through firecrown

The three fits of `fit_velocity.ipynb`, with $P_{\theta\theta}(k)$ from pyCCL and the
likelihood sampled through firecrown:

- flip builds the data vectors, the covariance and the likelihood, as in
  `fit_velocity.ipynb`;
- `flip.covariance.firecrown_interface.FlipLikelihood` turns the flip likelihood
  into a firecrown likelihood; each evaluation goes through the firecrown
  update / prepare / reset cycle of the connectors, minimised with Minuit.

The last section fits the true velocities with the samplers firecrown connects
to (CosmoSIS, Cobaya).

Needs firecrown (Python >= 3.12).

In [1]:
from pathlib import Path

import iminuit
import numpy as np
import pandas as pd
import pyccl

from flip import __flip_dir_path__, data_vector, utils
from flip.covariance import covariance, fitter, firecrown_interface
from flip.power_spectra import compute_power_spectra

from firecrown.likelihood import NamedParameters
from firecrown.modeling_tools import CCLFactory, ModelingTools
from firecrown.updatable import ParamsMap, get_default_params

data_path = Path(__flip_dir_path__) / "data"

# SN data

In [2]:
data_velocity = pd.read_parquet(data_path / "data_velocity.parquet")
print("number of SNe:", len(data_velocity))

DataTrueVel = data_vector.DirectVel(
    {**data_velocity.to_dict(orient="list"), "velocity_error": np.zeros(len(data_velocity))}
)
DataVel = data_vector.snia_vectors.VelFromSALTfit(
    data_velocity.to_dict(orient="list"), velocity_estimator="full", h=0.70
)
coordinates_velocity = np.array(
    [data_velocity["ra"], data_velocity["dec"], data_velocity["rcom_zobs"]]
)

number of SNe: 518


# Power spectrum and covariance

$P_{\theta\theta}(k)$ from pyCCL (CAMB transfer function), normalised by
$\sigma_8^2$ so that the covariance amplitude is $f\sigma_8$.

In [3]:
ccl_cosmology = pyccl.Cosmology(
    Omega_c=0.25, Omega_b=0.05, h=0.70, n_s=0.96, sigma8=0.819,
    m_nu=0.06, mass_split="single", transfer_function="boltzmann_camb",
)
ktt, _, _, ptt, _ = compute_power_spectra(
    "pyccl_engine", ccl_cosmology, 0.0, 0.0005, 1.0, 1000,
    normalization_power_spectrum="growth_amplitude", power_spectrum_model="linearbel",
)


def covariance_velocity(sigma_u):
    return covariance.CovMatrix.init_from_flip(
        "carreres23",
        "velocity",
        {"vv": [[ktt, ptt * utils.Du(ktt, sigma_u) ** 2]]},
        coordinates_velocity=coordinates_velocity,
        size_batch=10_000,
        number_worker=8,
    )


covariance_fit = covariance_velocity(15.0)
sigmau_list = np.linspace(10.0, 20.0, 10)
covariance_list = [covariance_velocity(sigma_u) for sigma_u in sigmau_list]

[ 000001.18 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.38e-01 seconds


[ 000001.54 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.46e-01 seconds


[ 000001.89 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.43e-01 seconds


[ 000002.27 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.71e-01 seconds


[ 000002.62 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.41e-01 seconds


...

[ 000003.35 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.49e-01 seconds


[ 000003.72 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.53e-01 seconds


[ 000004.09 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.61e-01 seconds


[ 000004.45 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.50e-01 seconds


[ 000004.81 ]: 10-30 09:50  root            INFO     Covariance matrix generated from flip with carreres23 model in 3.53e-01 seconds


# firecrown likelihood and minimiser

firecrown samplers maximise the log-likelihood: the flip likelihood is built with
`negative_log_likelihood=False`. `FlipLikelihood` registers its parameters as
firecrown parameters `flip_<name>`. The cosmological parameters are those of the
firecrown `ModelingTools`; they do not enter the likelihood here.

`loglike` is one sampler step, as in the connectors. Minuit is set up as in
flip's `FitMinuit` (start, step, limits, errordef).

In [4]:
likelihood_properties = {"inversion_method": "cholesky", "negative_log_likelihood": False}
cosmology_values = {"Omega_c": 0.25, "Omega_b": 0.05, "h": 0.70, "n_s": 0.96,
                    "sigma8": 0.819, "m_nu": 0.06}


def firecrown_likelihood(flip_likelihood):
    likelihood = firecrown_interface.FlipLikelihood(flip_likelihood, parameter_prefix="flip")
    tools = ModelingTools(ccl_factory=CCLFactory(creation_mode="pure_ccl_mode", mass_split="single"))
    return likelihood, tools


def loglike(likelihood, tools, values):
    params = ParamsMap({**get_default_params(likelihood, tools), **values})
    likelihood.update(params)
    tools.update(params)
    tools.prepare()
    try:
        return likelihood.compute_loglike_for_sampling(tools)
    finally:
        likelihood.reset()
        tools.reset()


def fit_firecrown(likelihood, tools, parameter_dict):
    names = list(parameter_dict)

    def minus_loglike(values):
        point = {f"flip_{n}": float(v) for n, v in zip(names, values)}
        return -loglike(likelihood, tools, {**cosmology_values, **point})

    minuit_fitter = fitter.FitMinuit(likelihood=minus_loglike)
    minuit_fitter.minuit = iminuit.Minuit(
        minus_loglike, [parameter_dict[n]["value"] for n in names], name=names
    )
    minuit_fitter.setup_minuit(parameter_dict)
    minuit_fitter.run(migrad=True, hesse=True)
    return minuit_fitter

# Fit true velocity

In [5]:
parameter_dict_true = {"fs8": {"value": 0.4,
                               "limit_low": 0.0,
                               "fixed": False,
                              },
                       "sigv": {"value": 200,
                               "limit_low": 0.0,
                               "fixed": False,
                              },
                      }

flip_likelihood_true = fitter.BaseFitter.select_likelihood("multivariate_gaussian").init_from_covariance(
    covariance_fit, DataTrueVel, list(parameter_dict_true), likelihood_properties=likelihood_properties
)
likelihood_true, tools_true = firecrown_likelihood(flip_likelihood_true)
print("firecrown parameters:", sorted(likelihood_true.required_parameters().get_params_names()))

fit_true = fit_firecrown(likelihood_true, tools_true, parameter_dict_true)
fit_true.minuit

firecrown parameters: ['flip_fs8', 'flip_sigv']


[ 000005.34 ]: 10-30 09:50  root            INFO     ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3778                       │              Nfcn = 62               │
│ EDM = 1.27e-07 (Goal: 0.0001)    │            time = 0.5 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────

[ 000005.39 ]: 10-30 09:50  root            INFO     ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3778                       │              Nfcn = 72               │
│ EDM = 1.27e-07 (Goal: 0.0001)    │            time = 0.5 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3778                       │              Nfcn = 72               │
│ EDM = 1.27e-07 (Goal: 0.0001)    │            time = 0.5 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ fs8  │   0.43    │   0.06    │            │            │    0    │         │       │
│ 1 │ sigv │    329    │    11     │            │            │    0    │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌──────┬─────────────────┐
│      │     fs8    sigv │
├──────┼─────────────────┤
│  fs8 │ 0.00323 -0.1082 │
│ sigv │ -0.1082     124 │
└──────┴─────────────────┘

# Fit true velocity interpolation sigma_u

In [6]:
parameter_dict_interp = {"fs8": {"value": 0.4,
                                 "limit_low": 0.0,
                                 "fixed": False,
                                },
                         "sigv": {"value": 200,
                                 "limit_low": 0.0,
                                 "fixed": False,
                                },
                         "sigu": {"value": 15.0,
                                 "limit_low": 13.0,
                                 "limit_up": 17.0,
                                 "fixed": False,
                                },
                        }

flip_likelihood_interp = fitter.BaseFitter.select_likelihood(
    "multivariate_gaussian_interp1d"
).init_from_covariance(
    covariance_list, DataTrueVel, list(parameter_dict_interp),
    likelihood_properties=likelihood_properties,
    interpolation_value_name="sigu", interpolation_value_range=sigmau_list,
)
likelihood_interp, tools_interp = firecrown_likelihood(flip_likelihood_interp)

fit_interp = fit_firecrown(likelihood_interp, tools_interp, parameter_dict_interp)
fit_interp.minuit

[ 000006.89 ]: 10-30 09:50  root            INFO     ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3780                       │              Nfcn = 85               │
│ EDM = 0.00022 (Goal: 0.0001)     │            time = 1.3 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────

[ 000007.11 ]: 10-30 09:50  root            INFO     ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3780                       │              Nfcn = 101              │
│ EDM = 0.000215 (Goal: 0.0001)    │            time = 1.5 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3780                       │              Nfcn = 101              │
│ EDM = 0.000215 (Goal: 0.0001)    │            time = 1.5 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ fs8  │   0.45    │   0.06    │            │            │    0    │         │       │
│ 1 │ sigv │    330    │    11     │            │            │    0    │         │       │
│ 2 │ sigu │   17.0    │    3.4    │            │            │   13    │   17    │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌──────┬──────────────────────┐
│      │    fs8   sigv   sigu │
├──────┼──────────────────────┤
│  fs8 │ 0.0037 -0.106  0.006 │
│ sigv │ -0.106    124    0.4 │
│ sigu │  0.006    0.4  0.657 │
└──────┴──────────────────────┘

# Fit estimate velocity

In [7]:
parameter_dict_salt = {"fs8": {"value": 0.4,
                               "limit_low": 0.0,
                               "limit_up": 1.0,
                               "fixed": False,
                              },
                       "sigv": {"value": 200,
                               "limit_low": 0.0,
                               "limit_up": 1000,
                               "fixed": False,
                              },
                       "alpha": {"value": 0.1,
                               "limit_low": 0.0,
                               "limit_up": 0.25,
                               "fixed": False,
                              },
                       "beta": {"value": 3.0,
                               "limit_low": 2.5,
                               "limit_up": 4.5,
                               "fixed": False,
                              },
                       "M_0": {"value": -19,
                               "limit_low": -21,
                               "limit_up": -18,
                               "fixed": False,
                              },
                       "sigma_M": {"value": 0.1,
                               "limit_low": 0.0,
                               "limit_up": 100.0,
                               "fixed": False,
                              }}

flip_likelihood_salt = fitter.BaseFitter.select_likelihood("multivariate_gaussian").init_from_covariance(
    covariance_fit, DataVel, list(parameter_dict_salt), likelihood_properties=likelihood_properties
)
likelihood_salt, tools_salt = firecrown_likelihood(flip_likelihood_salt)

fit_salt = fit_firecrown(likelihood_salt, tools_salt, parameter_dict_salt)
fit_salt.minuit

/global/cfs/cdirs/m1727/ravouxco/ccl_dev/conda_env/lib/python3.12/site-packages/iminuit/util.py:133: IMinuitWarning: Assigned errors must be positive. Non-positive values are replaced by a heuristic.
  self._set(index, value)


[ 000008.59 ]: 10-30 09:50  root            INFO     ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 4088                       │              Nfcn = 330              │
│ EDM = 8.81e-06 (Goal: 0.0001)    │            time = 1.5 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬─────────┬───────────┬────────

[ 000008.79 ]: 10-30 09:50  root            INFO     ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 4088                       │              Nfcn = 376              │
│ EDM = 9.06e-06 (Goal: 0.0001)    │            time = 1.7 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬─────────┬───────────┬────────

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 4088                       │              Nfcn = 376              │
│ EDM = 9.06e-06 (Goal: 0.0001)    │            time = 1.7 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│     SOME parameters at limit     │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬─────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name    │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼─────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ fs8     │   0.37    │   0.10    │            │            │    0    │    1    │       │
│ 1 │ sigv    │     0     │   0.7e3   │            │            │    0    │  1000   │       │
│ 2 │ alpha   │   0.127   │   0.007   │            │            │    0    │  0.25   │       │
│ 3 │ beta    │   3.09    │   0.07    │            │            │   2.5   │   4.5   │       │
│ 4 │ M_0     │  -19.063  │   0.022   │            │            │   -21   │   -18   │       │
│ 5 │ sigma_M │   0.140   │   0.005   │            │            │    0    │   100   │       │
└───┴─────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────────┬───────────────────────────────────────────────────────────────────┐
│         │        fs8       sigv      alpha       beta        M_0    sigma_M │
├─────────┼───────────────────────────────────────────────────────────────────┤
│     fs8 │     0.0101      0.024   -0.01e-3     -0.001     0.2e-3  -0.110e-3 │
│    sigv │      0.024        152   -0.86e-3     -0.005     0.8e-3 -24.290e-3 │
│   alpha │   -0.01e-3   -0.86e-3   4.71e-05   -0.02e-3         -0          0 │
│    beta │     -0.001     -0.005   -0.02e-3     0.0049    -0.3e-3   0.008e-3 │
│     M_0 │     0.2e-3     0.8e-3         -0    -0.3e-3   0.000468  -0.003e-3 │
│ sigma_M │  -0.110e-3 -24.290e-3          0   0.008e-3  -0.003e-3   2.63e-05 │
└─────────┴───────────────────────────────────────────────────────────────────┘

# True velocity fit with the DESC samplers

The true-velocity likelihood, fitted by the samplers firecrown connects to:
CosmoSIS (`maxlike`, Nelder-Mead) and Cobaya (`minimize`, iminuit), run from this
Python process. The flip likelihood is registered with `set_flip_likelihood`; both
load it with the firecrown factory
`likelihood_source = flip.covariance.firecrown_interface`, whose `build_likelihood`
returns the registered likelihood (build parameter `flip_likelihood`).

The cosmological parameters are fixed; only `flip_fs8` and `flip_sigv` are fitted.

In [8]:
import cobaya
import firecrown
from cosmosis import Inifile, run_cosmosis
from firecrown.connector.cobaya.likelihood import LikelihoodConnector

firecrown_interface.set_flip_likelihood(flip_likelihood_true, name="flip_true")
build_parameters = {"flip_likelihood": "flip_true", "ccl_creation_mode": "pure_ccl_mode",
                    "mass_split": "single"}

# Fixed cosmology: CCL parameters of the firecrown ModelingTools.
fixed_parameters = {
    name: value
    for name, value in {**get_default_params(likelihood_true, tools_true), **cosmology_values}.items()
    if not name.startswith("flip_")
}
start = {name: parameter_dict_true[name]["value"] for name in parameter_dict_true}
bounds = {"fs8": (0.0, 1.0), "sigv": (0.0, 1000.0)}

## CosmoSIS

In [9]:
def fit_cosmosis():
    config = {
        "runtime": {"sampler": "maxlike", "verbosity": "quiet"},
        "maxlike": {"method": "Nelder-Mead", "tolerance": 1e-6, "maxiter": 10000},
        "pipeline": {"modules": "firecrown_likelihood", "likelihoods": "firecrown"},
        "firecrown_likelihood": {
            "file": str(Path(firecrown.__file__).parent / "connector" / "cosmosis" / "likelihood.py"),
            "likelihood_source": "flip.covariance.firecrown_interface",
            "sampling_parameters_sections": "cosmological_parameters flip",
            **build_parameters,
        },
    }
    values = {
        "cosmological_parameters": fixed_parameters,
        "flip": {f"flip_{n}": f"{bounds[n][0]} {start[n]} {bounds[n][1]}" for n in start},
    }
    _, table = run_cosmosis(config, values=Inifile(values), output="astropy")
    best = table[np.argmax(table["like"])]
    return {**{n: best[f"flip--flip_{n}"] for n in start}, "ln L": best["like"]}


cosmosis_fit = fit_cosmosis()

Setting up pipeline from pre-constructed configuration
------------------------------------------------------

Setting up module firecrown_likelihood
---------------------------------------


Setup all pipeline modules




Parameter Priors
----------------
cosmological_parameters--neff     ~ delta(3.044)
cosmological_parameters--t_cmb    ~ delta(2.7255)
cosmological_parameters--omega_b  ~ delta(0.05)
cosmological_parameters--omega_k  ~ delta(0.0)
cosmological_parameters--m_nu     ~ delta(0.06)
cosmological_parameters--w0       ~ delta(-1.0)
cosmological_parameters--wa       ~ delta(0.0)
cosmological_parameters--sigma8   ~ delta(0.819)
cosmological_parameters--n_s      ~ delta(0.96)
cosmological_parameters--h        ~ delta(0.7)
cosmological_parameters--omega_c  ~ delta(0.25)
flip--flip_fs8                    ~ U(0.0, 1.0)
flip--flip_sigv                   ~ U(0.0, 1000.0)

****************************************************
* Running sampler 1/1: maxlike
* Running in serial mode.
****************************************************
--------------------------------------------------


NOTE: Running optimizer in **max-like** mode:


NOTE: not including the prior, just the likelihood.


...

[Rank 0] Best fit (by likelihood):
0.4339342010478299   328.6787944378994


[Rank 0] Posterior: -3784.870059359427


[Rank 0] Likelihood: -3777.962304080445



Total posterior evaluations = 92 across all processes


Successful posterior evaluations = 92 across all processes


## Cobaya

firecrown's `LikelihoodConnector.logp` expects Cobaya to always request the
derived parameters; `minimize` does not (`return_derived=False`). The subclass
below passes an empty dictionary in that case.

In [10]:
class CobayaConnector(LikelihoodConnector):
    def logp(self, _derived=None, **params_values):
        return super().logp(_derived={} if _derived is None else _derived, **params_values)


info = {
    "likelihood": {
        "flip": {
            "external": CobayaConnector,
            "firecrownIni": "flip.covariance.firecrown_interface",
            "build_parameters": NamedParameters(build_parameters),
        }
    },
    "params": {
        **fixed_parameters,
        **{
            f"flip_{n}": {"prior": {"min": bounds[n][0], "max": bounds[n][1]}, "ref": start[n],
                          "proposal": 0.1 * start[n]}
            for n in start
        },
    },
    "sampler": {"minimize": {"method": "iminuit"}},
    "stop_at_error": True,
}
_, cobaya_minimizer = cobaya.run(info)
minimum = cobaya_minimizer.products()["minimum"]
cobaya_fit = {**{n: minimum[f"flip_{n}"] for n in start}, "ln L": -0.5 * minimum["chi2"]}

[ 000013.10 ]: 10-30 09:50  minimize        INFO     Initializing


[minimize] Initializing


[ 000013.11 ]: 10-30 09:50  minimize        INFO     Run 1/2 will start from random initial point:


[minimize] Run 1/2 will start from random initial point:


[ 000013.11 ]: 10-30 09:50  minimize        INFO     {'flip_fs8': np.float64(0.4), 'flip_sigv': np.float64(200.0)}


...

[minimize] -log(posterior) minimized to 3784.87


[ 000013.68 ]: 10-30 09:50  minimize        INFO     Parameter values at minimum:
   weight  minuslogpost  flip_fs8   flip_sigv  minuslogprior  minuslogprior__0         chi2   chi2__flip
0     1.0   3784.870064  0.433866  328.649012       6.907755          6.907755  7555.924618  7555.924618


[minimize] Parameter values at minimum:
   weight  minuslogpost  flip_fs8   flip_sigv  minuslogprior  minuslogprior__0         chi2   chi2__flip
0     1.0   3784.870064  0.433866  328.649012       6.907755          6.907755  7555.924618  7555.924618


[ 000013.68 ]: 10-30 09:50  minimize        INFO     Full set of minima:
{'0': (3784.87006436174, True), '1': (3784.87006436174, True)}


[minimize] Full set of minima:
{'0': (3784.87006436174, True), '1': (3784.87006436174, True)}


## Comparison

In [11]:
pd.DataFrame(
    {
        "firecrown + Minuit": {**fit_true.minuit.values.to_dict(), "ln L": -fit_true.minuit.fval},
        "CosmoSIS": cosmosis_fit,
        "Cobaya": cobaya_fit,
    }
).T

,fs8,sigv,ln L
firecrown + Minuit,0.433958,328.681143,-3777.962304
CosmoSIS,0.433934,328.678794,-3777.962304
Cobaya,0.433866,328.649012,-3777.962309
